# v7 CPU finalization — frozen cap-15 confirmation (direct / crash-safe style)

This is the **same frozen finalization experiment** as the previous `v7_cpu_finalize` notebook, but it runs **directly in the notebook kernel** rather than generating and launching a separate Python runner.

Nothing in the experiment specification is changed:

- architecture: current v7 shared 4×50 ReLU trunk + maturity Softplus heads;
- loss: ordinary MSE;
- rolling recalibration: **exactly 15 SDE updates/day**;
- 10 SDE optimizer steps + 10 control-variate/hedge optimizer steps per update;
- 4,096 paths/step, 400 sampled contracts/step;
- learning rates unchanged from v7 (`1e-4` SDE, `1e-3` hedge);
- no patience-based stopping and no same-day checkpoint reranking;
- primary confirmation evaluation: **100k common-shock paths**;
- untouched test period: **Mar-8 → Mar-28**, 14 next-day transitions;
- robustness pass: **400k antithetic paths**, evaluation only, with the same trained models.

The only change is execution/visibility: the run is in-kernel, with notebook-native progress bars around the long loops. Existing checkpoints/results are still reused.

In [1]:
import gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import search_pipeline as sp
import overnight_fixed_runner as ofr
from audit_v7 import load_notebook_environment

# Load the exact v7 notebook environment used by the frozen-search code.
ns = load_notebook_environment()

# Keep the frozen CPU-final settings exactly as before.
ns["FINAL_PATH_CHUNK"] = 2000
torch.set_num_threads(3)

# Make the crash-safe notebook's stage messages coexist cleanly with tqdm.
ns["stage_log"] = lambda message: tqdm.write(str(message))

print("Frozen v7 environment loaded directly into this kernel.")
print("PyTorch threads:", torch.get_num_threads())
print("FINAL_PATH_CHUNK:", ns["FINAL_PATH_CHUNK"])
print("Run root:", sp.RUN_ROOT)

Device: cpu | PyTorch threads: 1
Compute profile: cpu | transitions: 5
Calibration universe: all filtered OTM
Objective: plain contract-level raw price MSE (spot-scaled)
Main budget: 100 SDE phases | polish: 30 SDE phases
LR: 0.0001 | hedge LR: 0.001 | scheduler: none
Early stop: 10 SDE phases without a new raw-RMSE best
options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809
Anchor selected/train/validation: 4304 3443 861
Anchor selected as fraction of filtered OTM: 100.0%
Anchor maturities train/validation: 36 36
Anchor training contracts: 3443
Anchor maturity heads: 36
Anchor simulation steps: 129
Frozen v7 environment loaded directly into this kernel.
PyTorch threads: 3
FINAL_PATH_CHUNK: 2000
Run root: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSD

## Helpers

These are orchestration-only helpers. They do not change model architecture, training loss, optimizer settings, seeds, data selection, path counts, or evaluation definitions.

In [2]:
def load_model(checkpoint_path):
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = ns["MaturityHeadNeuralLocalVol"](payload["boundaries"]).cpu()
    model.load_state_dict(payload["final_model_state"], strict=True)
    model.eval()
    sp.set_grad(model, False)
    return model


def existing_row(path, date):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if "date" not in df.columns:
        return None
    df["date"] = pd.to_datetime(df["date"])
    hit = df.loc[df["date"] == pd.Timestamp(date)]
    return None if hit.empty else hit.iloc[-1].to_dict()


def pooled(df):
    n = df["otm_n"].to_numpy(float)
    p = df["otm_pricing_RMSE"].to_numpy(float)
    h = df["otm_NLV_hedge_RMSE"].to_numpy(float)
    m = df["otm_market_delta_hedge_RMSE"].to_numpy(float)
    out = {
        "otm_n": int(n.sum()),
        "pooled_otm_pricing_RMSE": float(np.sqrt(np.sum(n * p**2) / np.sum(n))),
        "pooled_otm_NLV_hedge_RMSE": float(np.sqrt(np.sum(n * h**2) / np.sum(n))),
        "pooled_otm_market_delta_hedge_RMSE": float(np.sqrt(np.sum(n * m**2) / np.sum(n))),
    }
    if "otm_pricing_MAE" in df.columns:
        out["weighted_otm_pricing_MAE"] = float(
            np.sum(n * df["otm_pricing_MAE"].to_numpy(float)) / np.sum(n)
        )
    return out


def train_fixed_endpoint_visible(**kwargs):
    """Run the unchanged fixed-endpoint trainer while exposing optimizer-step progress.

    The wrapper temporarily instruments simulate_estimators only to advance tqdm.
    The original numerical function is called unchanged and restored immediately after.
    """
    updates = int(kwargs["updates"])
    steps_per_phase = int(ns.get("STEPS_PER_PHASE", 10))
    total_steps = updates * 2 * steps_per_phase
    date = pd.Timestamp(kwargs["date"])

    original_simulate = ns.get("simulate_estimators")
    bar = tqdm(
        total=total_steps,
        desc=f"Train {date:%Y-%m-%d} | cap-{updates} MSE",
        unit="opt-step",
        dynamic_ncols=True,
        mininterval=0.5,
        leave=True,
    )

    if original_simulate is not None:
        def visible_simulate(*args, **inner_kwargs):
            result = original_simulate(*args, **inner_kwargs)
            bar.update(1)
            return result
        ns["simulate_estimators"] = visible_simulate

    try:
        result = ofr.train_fixed_endpoint(ns, **kwargs)
        if bar.n < bar.total:
            # If the trainer resumed internally or uses a path not routed through
            # simulate_estimators, complete only the display bar; this changes no computation.
            bar.update(bar.total - bar.n)
        return result
    finally:
        if original_simulate is not None:
            ns["simulate_estimators"] = original_simulate
        bar.close()

## 1. Frozen 14-transition confirmation — 15 updates/day, 100k evaluation

This is the primary untouched test. Re-running the cell is safe: completed endpoint checkpoints and completed transition summaries are reused.

In [3]:
# Exact same development anchor as the previous finalization notebook.
dev_dir = sp.RUN_ROOT / "capped15_endpoint"
dev_last = dev_dir / "20240307_checkpoint.pt"
if not dev_last.exists():
    raise FileNotFoundError(
        f"Missing {dev_last}. Run/extract the completed capped15_endpoint development branch first."
    )

out = sp.RUN_ROOT / "frozen_test_capped15_endpoint"
out.mkdir(parents=True, exist_ok=True)

previous = load_model(dev_last)
previous_date = pd.Timestamp("2024-03-07")

trading_dates = [pd.Timestamp(x) for x in ns["trading_dates"]]
start = trading_dates.index(pd.Timestamp("2024-03-08"))
training_dates = trading_dates[start:-1]
target_dates = trading_dates[start+1:]
assert len(training_dates) == 14 and len(target_dates) == 14

partial_summary = out / "unseen_summary_partial.csv"
partial_meta = out / "training_metadata_partial.csv"
summaries, metadata = [], []

transitions = list(zip(training_dates, target_dates))
outer = tqdm(
    transitions,
    total=len(transitions),
    desc="Frozen test: train + 100k eval",
    unit="transition",
    dynamic_ncols=True,
    mininterval=0.5,
    leave=True,
)

for index, (date, target) in enumerate(outer, start=1):
    outer.set_description(f"Frozen {date:%m-%d} → {target:%m-%d}")
    outer.set_postfix_str("checkpoint")

    cp = out / f"{date:%Y%m%d}_checkpoint.pt"
    old_summary = existing_row(partial_summary, date)
    old_meta = existing_row(partial_meta, date)

    if cp.exists():
        outer.set_postfix_str("reuse trained endpoint")
        tqdm.write(f"[reuse checkpoint] {date.date()}")
        model = load_model(cp)
        ctx = None

        if old_meta is None:
            payload = torch.load(cp, map_location="cpu", weights_only=False)
            meta = {
                "date": date,
                "endpoint_update": int(payload["endpoint_update"]),
                "objective": "mse",
                "paths_per_step": 4096,
                "elapsed_seconds": np.nan,
            }
            del payload
        else:
            meta = old_meta
    else:
        outer.set_postfix_str("training: cap-15 MSE")
        tqdm.write(f"\nTRAIN {date.date()} → {target.date()} | fixed cap-15 MSE")

        model, ctx, meta = train_fixed_endpoint_visible(
            name="frozen_test_capped15_endpoint",
            objective="mse",
            updates=15,
            paths_per_step=4096,
            seed_offset=0,
            date=date,
            previous_model=previous,
            previous_date=previous_date,
            output_dir=out,
        )

    if old_summary is None:
        outer.set_postfix_str("100k common-shock eval")
        tqdm.write(f"EVAL 100k {date.date()} → {target.date()}")

        cfg = sp.SearchConfig(
            name="frozen_test_capped15_endpoint",
            eval_paths=100000,
            seed_offset=0,
        )
        summary = sp.evaluate_one_transition(
            ns, model, date, target, cfg, out, index
        )
    else:
        outer.set_postfix_str("reuse 100k result")
        tqdm.write(f"[reuse 100k] {date.date()} → {target.date()}")
        summary = old_summary

    summaries.append(summary)
    metadata.append(meta)

    # Same crash-safe partial persistence as before.
    pd.DataFrame(summaries).to_csv(partial_summary, index=False)
    pd.DataFrame(metadata).to_csv(partial_meta, index=False)

    if ctx is not None:
        del ctx
    del previous
    previous = model
    previous_date = date
    gc.collect()

outer.close()

primary_summary = pd.DataFrame(summaries)
primary_summary.to_csv(out / "unseen_summary.csv", index=False)
pd.DataFrame(metadata).to_csv(out / "training_metadata.csv", index=False)

primary_pooled = pooled(primary_summary)
pd.DataFrame([
    {"variant": "frozen_test_capped15_endpoint", **primary_pooled}
]).to_csv(out / "pooled_summary.csv", index=False)

print("\nPRIMARY FROZEN CONFIRMATION COMPLETE")
print(primary_pooled)

Frozen test: train + 100k eval:   0%|          | 0/14 [00:00<?, ?transition/s]


TRAIN 2024-03-08 → 2024-03-11 | fixed cap-15 MSE


Train 2024-03-08 | cap-15 MSE:   0%|          | 0/300 [00:00<?, ?opt-step/s]

Warm-start 2024-03-07 -> 2024-03-08: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y
[resume] frozen_test_capped15_endpoint 2024-03-08 from update 8/15


[checkpoint] frozen_test_capped15_endpoint 2024-03-08 10/15 | train=4.20858e-07


...

transition: begin
transition AM=0: simulation begin, rows=6038


Future-date MC AM=0: 100%|██████████| 100000/100000 [00:10<00:00, 9384.27path/s]


transition AM=0: simulation returned; terminal storage=10.7 MiB
transition AM=0: payoff begin T=0.0410958904109589, contracts=169
transition AM=0: payoff begin T=0.049315068493150684, contracts=164
transition AM=0: payoff begin T=0.052054794520547946, contracts=161
transition AM=0: payoff begin T=0.0547945205479452, contracts=168
transition AM=0: payoff begin T=0.057534246575342465, contracts=158
transition AM=0: payoff begin T=0.06027397260273973, contracts=231
transition AM=0: payoff begin T=0.0684931506849315, contracts=148
transition AM=0: payoff begin T=0.07123287671232877, contracts=145
transition AM=0: payoff begin T=0.07397260273972603, contracts=99
transition AM=0: payoff begin T=0.07671232876712329, contracts=99
transition AM=0: payoff begin T=0.07945205479452055, contracts=247
transition AM=0: payoff begin T=0.08767123287671233, contracts=96
transition AM=0: payoff begin T=0.09041095890410959, contracts=272
transition AM=0: payoff begin T=0.09315068493150686, contracts=68
tr

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:10<00:00, 9421.30path/s]


transition AM=1: simulation returned; terminal storage=4.6 MiB
transition AM=1: payoff begin T=0.06027397260273973, contracts=235
transition AM=1: payoff begin T=0.136986301369863, contracts=370
transition AM=1: payoff begin T=0.2328767123287671, contracts=429
transition AM=1: payoff begin T=0.3095890410958904, contracts=447
transition AM=1: payoff begin T=0.3863013698630137, contracts=366
transition AM=1: payoff begin T=0.4821917808219178, contracts=360
transition AM=1: payoff begin T=0.5589041095890411, contracts=152
transition AM=1: payoff begin T=0.6356164383561644, contracts=156
transition AM=1: payoff begin T=0.7315068493150685, contracts=159
transition AM=1: payoff begin T=0.8082191780821918, contracts=158
transition AM=1: payoff begin T=0.9041095890410958, contracts=161
transition AM=1: payoff begin T=0.9808219178082191, contracts=171
transition AM=1: pricing complete
transition: concatenate begin
transition: delta merge begin
transition: hedge calculations complete; OTM flags 

## 2. Primary untouched-test result

In [4]:
primary = sp.RUN_ROOT / "frozen_test_capped15_endpoint"
display(pd.read_csv(primary / "pooled_summary.csv"))
display(pd.read_csv(primary / "unseen_summary.csv"))

,variant,otm_n,pooled_otm_pricing_RMSE,pooled_otm_NLV_hedge_RMSE,pooled_otm_market_delta_hedge_RMSE,weighted_otm_pricing_MAE
0,frozen_test_capped15_endpoint,57622,3.267747,1.90165,1.321281,2.547263


,date,next_date,full_n,full_pricing_RMSE,full_pricing_MAE,full_pricing_Rel_MAE_pct,full_pricing_inside_bid_ask,full_NLV_hedge_RMSE,full_NLV_hedge_MAE,full_NLV_hedge_bias,...,otm_pricing_Rel_MAE_pct,otm_pricing_inside_bid_ask,otm_NLV_hedge_RMSE,otm_NLV_hedge_MAE,otm_NLV_hedge_bias,otm_market_delta_hedge_RMSE,otm_market_delta_hedge_MAE,otm_market_delta_hedge_bias,otm_NLV_minus_market_delta_RMSE,otm_delta_correlation
0,2024-03-08,2024-03-11,9350,6.651689,5.244914,5.085737,0.070588,2.474795,1.949630,-1.511464,...,8.025629,0.055397,1.201294,0.847895,-0.646012,1.323909,1.009280,-0.935654,0.055532,0.996961
1,2024-03-11,2024-03-12,9014,9.790324,7.752355,7.074710,0.025738,3.020835,2.763269,-2.719904,...,12.226235,0.042857,3.437819,3.208108,-3.208108,0.525617,0.388304,-0.201071,0.057770,0.996683
2,2024-03-12,2024-03-13,9001,7.253985,5.770221,4.396819,0.048328,1.257737,1.030494,-0.879322,...,6.188780,0.045397,1.141324,0.917889,-0.909276,1.518453,1.378353,-1.378347,0.051834,0.996920
3,2024-03-13,2024-03-14,8781,5.797954,4.522632,4.599025,0.103405,2.908011,2.604300,0.218515,...,7.626990,0.058484,2.265276,2.009310,1.540507,1.633431,1.498804,0.841765,0.052453,0.997046
4,2024-03-14,2024-03-15,9363,3.697469,2.926120,4.129225,0.310584,2.047921,1.742414,1.430131,...,7.472696,0.048331,1.829788,1.576973,1.546592,0.669306,0.532054,-0.243718,0.057803,0.997235
5,2024-03-15,2024-03-18,9178,6.412237,4.955359,5.262056,0.082916,2.280938,2.143032,-2.131471,...,8.773566,0.070260,2.266355,2.169959,-2.169959,0.574983,0.444559,-0.370135,0.059712,0.997275
6,2024-03-18,2024-03-19,9454,8.712558,6.754864,5.307438,0.033425,2.722517,2.215366,-1.342203,...,7.769374,0.068457,2.712675,2.304301,-2.304181,1.217099,0.965505,-0.746558,0.058009,0.997205
7,2024-03-19,2024-03-20,8781,7.243538,5.863554,7.538713,0.089398,4.868955,3.969643,-3.397891,...,13.756388,0.036391,2.117756,1.839738,-1.806216,1.621243,1.404857,0.466957,0.054038,0.996796
8,2024-03-20,2024-03-21,8904,8.209722,6.595259,6.197063,0.027965,2.432036,2.064862,0.166016,...,9.845190,0.028372,1.522764,1.326812,-0.984319,1.073490,0.773970,-0.149355,0.053971,0.996844
9,2024-03-21,2024-03-22,9601,7.923974,6.126389,4.095208,0.078429,1.225765,0.976400,-0.293397,...,4.675347,0.166190,0.913125,0.695609,-0.673292,1.153568,1.008765,-1.006807,0.050658,0.996931


## 3. 400k-path evaluation-only robustness check

This reuses the exact same trained checkpoints. There is **no retraining and no reopening of model selection**.

In [5]:
src = sp.RUN_ROOT / "frozen_test_capped15_endpoint"
if not (src / "unseen_summary.csv").exists():
    raise FileNotFoundError("Run the primary 100k frozen confirmation first.")

out400 = sp.RUN_ROOT / "frozen_test_capped15_endpoint_eval400k"
out400.mkdir(parents=True, exist_ok=True)

trading_dates = [pd.Timestamp(x) for x in ns["trading_dates"]]
start = trading_dates.index(pd.Timestamp("2024-03-08"))
training_dates = trading_dates[start:-1]
target_dates = trading_dates[start+1:]
assert len(training_dates) == 14 and len(target_dates) == 14

partial400 = out400 / "unseen_summary_400k_partial.csv"
summaries400 = []

transitions400 = list(zip(training_dates, target_dates))
outer400 = tqdm(
    transitions400,
    total=len(transitions400),
    desc="400k robustness eval",
    unit="transition",
    dynamic_ncols=True,
    mininterval=0.5,
    leave=True,
)

for index, (date, target) in enumerate(outer400, start=1):
    outer400.set_description(f"400k {date:%m-%d} → {target:%m-%d}")
    outer400.set_postfix_str("checkpoint/result")

    old = existing_row(partial400, date)
    if old is not None:
        outer400.set_postfix_str("reuse 400k result")
        summaries400.append(old)
        tqdm.write(f"[reuse 400k] {date.date()} → {target.date()}")
        continue

    cp = src / f"{date:%Y%m%d}_checkpoint.pt"
    model = load_model(cp)

    cfg = sp.SearchConfig(
        name="frozen_test_capped15_endpoint_eval400k",
        eval_paths=400000,
        seed_offset=0,
    )

    outer400.set_postfix_str("400k antithetic eval")
    tqdm.write(f"\nEVAL 400k {date.date()} → {target.date()}")
    s = sp.evaluate_one_transition(ns, model, date, target, cfg, out400, index)

    summaries400.append(s)
    pd.DataFrame(summaries400).to_csv(partial400, index=False)

    del model
    gc.collect()

outer400.close()

eval400_summary = pd.DataFrame(summaries400)
eval400_summary.to_csv(out400 / "unseen_summary_400k.csv", index=False)

eval400_pooled = pooled(eval400_summary)
pd.DataFrame([{
    "variant": "frozen_test_capped15_endpoint_eval400k",
    **eval400_pooled,
}]).to_csv(out400 / "pooled_summary_400k.csv", index=False)

print("\n400K ROBUSTNESS EVALUATION COMPLETE")
print(eval400_pooled)

400k robustness eval:   0%|          | 0/14 [00:00<?, ?transition/s]


EVAL 400k 2024-03-08 → 2024-03-11


Same-day MC: 100%|██████████| 400000/400000 [00:41<00:00, 9568.91path/s]


transition: begin
transition AM=0: simulation begin, rows=6554


Future-date MC AM=0: 100%|██████████| 400000/400000 [00:39<00:00, 10101.05path/s]


...

transition: begin
transition AM=0: simulation begin, rows=6038


Future-date MC AM=0: 100%|██████████| 400000/400000 [00:37<00:00, 10558.00path/s]


transition AM=0: simulation returned; terminal storage=42.7 MiB
transition AM=0: payoff begin T=0.0410958904109589, contracts=169
transition AM=0: payoff begin T=0.049315068493150684, contracts=164
transition AM=0: payoff begin T=0.052054794520547946, contracts=161
transition AM=0: payoff begin T=0.0547945205479452, contracts=168
transition AM=0: payoff begin T=0.057534246575342465, contracts=158
transition AM=0: payoff begin T=0.06027397260273973, contracts=231
transition AM=0: payoff begin T=0.0684931506849315, contracts=148
transition AM=0: payoff begin T=0.07123287671232877, contracts=145
transition AM=0: payoff begin T=0.07397260273972603, contracts=99
transition AM=0: payoff begin T=0.07671232876712329, contracts=99
transition AM=0: payoff begin T=0.07945205479452055, contracts=247
transition AM=0: payoff begin T=0.08767123287671233, contracts=96
transition AM=0: payoff begin T=0.09041095890410959, contracts=272
transition AM=0: payoff begin T=0.09315068493150686, contracts=68
tr

Future-date MC AM=1: 100%|██████████| 400000/400000 [00:39<00:00, 10221.48path/s]


transition AM=1: simulation returned; terminal storage=18.3 MiB
transition AM=1: payoff begin T=0.06027397260273973, contracts=235
transition AM=1: payoff begin T=0.136986301369863, contracts=370
transition AM=1: payoff begin T=0.2328767123287671, contracts=429
transition AM=1: payoff begin T=0.3095890410958904, contracts=447
transition AM=1: payoff begin T=0.3863013698630137, contracts=366
transition AM=1: payoff begin T=0.4821917808219178, contracts=360
transition AM=1: payoff begin T=0.5589041095890411, contracts=152
transition AM=1: payoff begin T=0.6356164383561644, contracts=156
transition AM=1: payoff begin T=0.7315068493150685, contracts=159
transition AM=1: payoff begin T=0.8082191780821918, contracts=158
transition AM=1: payoff begin T=0.9041095890410958, contracts=161
transition AM=1: payoff begin T=0.9808219178082191, contracts=171
transition AM=1: pricing complete
transition: concatenate begin
transition: delta merge begin
transition: hedge calculations complete; OTM flags

## 4. 400k robustness result

In [6]:
eval400 = sp.RUN_ROOT / "frozen_test_capped15_endpoint_eval400k"
display(pd.read_csv(eval400 / "pooled_summary_400k.csv"))
display(pd.read_csv(eval400 / "unseen_summary_400k.csv"))


,variant,otm_n,pooled_otm_pricing_RMSE,pooled_otm_NLV_hedge_RMSE,pooled_otm_market_delta_hedge_RMSE,weighted_otm_pricing_MAE
0,frozen_test_capped15_endpoint_eval400k,57622,3.22084,1.900956,1.321281,2.525278


,date,next_date,full_n,full_pricing_RMSE,full_pricing_MAE,full_pricing_Rel_MAE_pct,full_pricing_inside_bid_ask,full_NLV_hedge_RMSE,full_NLV_hedge_MAE,full_NLV_hedge_bias,...,otm_pricing_Rel_MAE_pct,otm_pricing_inside_bid_ask,otm_NLV_hedge_RMSE,otm_NLV_hedge_MAE,otm_NLV_hedge_bias,otm_market_delta_hedge_RMSE,otm_market_delta_hedge_MAE,otm_market_delta_hedge_bias,otm_NLV_minus_market_delta_RMSE,otm_delta_correlation
0,2024-03-08,2024-03-11,9350,6.659587,5.241532,5.058448,0.074545,2.475830,1.950422,-1.511878,...,7.947851,0.064670,1.200973,0.847951,-0.646126,1.323909,1.009280,-0.935654,0.055489,0.996971
1,2024-03-11,2024-03-12,9014,9.760928,7.744918,7.230043,0.020856,3.016337,2.762689,-2.718986,...,12.636815,0.032275,3.430437,3.207325,-3.207325,0.525617,0.388304,-0.201071,0.057634,0.996753
2,2024-03-12,2024-03-13,9001,7.143200,5.626863,4.123984,0.051105,1.256858,1.028806,-0.878062,...,5.586019,0.055288,1.140895,0.916533,-0.908145,1.518453,1.378353,-1.378347,0.051999,0.996907
3,2024-03-13,2024-03-14,8781,5.796688,4.547357,4.714147,0.101355,2.908517,2.604244,0.217395,...,7.895190,0.054719,2.263150,2.006643,1.539505,1.633431,1.498804,0.841765,0.052353,0.997057
4,2024-03-14,2024-03-15,9363,4.216070,3.329680,4.779193,0.295525,2.050136,1.744027,1.432095,...,8.711151,0.037124,1.834606,1.579049,1.548091,0.669306,0.532054,-0.243718,0.057929,0.997194
5,2024-03-15,2024-03-18,9178,6.459276,5.030825,5.395251,0.082480,2.277148,2.140766,-2.129348,...,9.083786,0.071704,2.264490,2.168418,-2.168418,0.574983,0.444559,-0.370135,0.059615,0.997281
6,2024-03-18,2024-03-19,9454,8.718397,6.739395,5.150226,0.028771,2.723518,2.216493,-1.339353,...,7.365064,0.061089,2.713020,2.301790,-2.301692,1.217099,0.965505,-0.746558,0.057984,0.997172
7,2024-03-19,2024-03-20,8781,7.184809,5.806411,7.346346,0.089284,4.886888,3.989347,-3.412705,...,13.300190,0.037149,2.125706,1.854075,-1.819687,1.621243,1.404857,0.466957,0.054398,0.996753
8,2024-03-20,2024-03-21,8904,8.057531,6.368274,5.665878,0.028976,2.431286,2.064771,0.166314,...,8.621218,0.036336,1.522318,1.327317,-0.983559,1.073490,0.773970,-0.149355,0.053914,0.996860
9,2024-03-21,2024-03-22,9601,7.900434,6.105467,4.089757,0.077596,1.225176,0.976085,-0.293995,...,4.678671,0.165476,0.912750,0.696000,-0.673779,1.153568,1.008765,-1.006807,0.050462,0.997012


In [7]:
from pathlib import Path
import zipfile

ROOT = Path.cwd()
OUT = ROOT / "v7_cpu_final_confirmation_results.zip"

folders = [
    sp.RUN_ROOT / "frozen_test_capped15_endpoint",
    sp.RUN_ROOT / "frozen_test_capped15_endpoint_eval400k",
]

with zipfile.ZipFile(OUT, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print("Missing:", folder)
            continue

        for p in folder.rglob("*"):
            if p.is_file():
                z.write(
                    p,
                    arcname=str(
                        Path(folder.name) / p.relative_to(folder)
                    )
                )

print("Created:", OUT)
print("Size MB:", round(OUT.stat().st_size / 1024**2, 1))

Created: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_run_workspace\v7_cpu_final_confirmation_results.zip
Size MB: 40.8
